## Example 1: Basic index creation and manipulation

In [ ]:
import ndindex
import numpy as np

print("Example 1: Basic index creation and manipulation")

try:
    print(f"ndindex version: {ndindex.__version__}")
    
    # Basic index types
    print(f"\nExploring basic index types:")
    
    # Create different types of indices
    indices = {
        'Integer': ndindex.Integer(5),
        'Slice': ndindex.Slice(2, 8, 2),
        'Tuple': ndindex.Tuple(ndindex.Integer(1), ndindex.Slice(None, 5)),
        # INSTANCES: `ndindex.ellipsis` is the class and `ndindex.newaxis` the submodule, neither an index.
        'Ellipsis': ndindex.ellipsis(),
        'Newaxis': ndindex.Newaxis()
    }
    
    for name, idx in indices.items():
        print(f"  {name}: {idx} (type: {type(idx).__name__})")
        print(f"    repr: {repr(idx)}")
        if hasattr(idx, 'raw'):
            print(f"    raw: {idx.raw}")
    # Known answers: Slice and Tuple are built on the compiled _Slice / _Tuple extension types,
    # so their argument normalisation, .args and .raw are what a miscompile would get wrong.
    assert ndindex.Slice.__mro__[1].__module__ == "ndindex._slice", ndindex.Slice.__mro__
    assert ndindex.Tuple.__mro__[1].__module__ == "ndindex._tuple", ndindex.Tuple.__mro__
    assert indices['Integer'].raw == 5, indices['Integer'].raw
    assert indices['Slice'].args == (2, 8, 2) and indices['Slice'].raw == slice(2, 8, 2), indices['Slice']
    assert ndindex.Slice(5).args == (None, 5, None), ndindex.Slice(5).args
    assert indices['Tuple'].raw == (1, slice(None, 5, None)), indices['Tuple'].raw
    assert indices['Tuple'].args == (ndindex.Integer(1), ndindex.Slice(None, 5, None)), indices['Tuple'].args
    for bad, error_type in [(lambda: ndindex.Slice(0, 5, 0), ValueError),
                            (lambda: ndindex.Tuple(..., ...), IndexError)]:
        try:
            bad()
        except error_type:
            pass
        else:
            raise AssertionError(f"ndindex accepted an invalid index; expected {error_type.__name__}")
    
    # Create sample array for indexing demonstrations
    print(f"\nCreating sample array for indexing:")
    arr = np.arange(24).reshape(4, 6)
    print(f"Array shape: {arr.shape}")
    print(f"Array content:")
    print(arr)
    
    # Apply different indices
    print(f"\nApplying different indices:")
    
    # Integer indexing
    int_idx = ndindex.Integer(2)
    result = arr[int_idx.raw]
    print(f"Integer index {int_idx}: {result} (shape: {result.shape})")
    assert (result == np.arange(12, 18)).all(), result
    
    # Slice indexing
    slice_idx = ndindex.Slice(1, 4, 2)
    result = arr[slice_idx.raw]
    print(f"Slice index {slice_idx}:")
    print(f"  Result shape: {result.shape}")
    print(f"  Result:\n{result}")
    
    # Tuple indexing (multidimensional)
    tuple_idx = ndindex.Tuple(ndindex.Slice(1, 3), ndindex.Slice(2, 5))
    result = arr[tuple_idx.raw]
    print(f"Tuple index {tuple_idx}:")
    assert result.tolist() == [[8, 9, 10], [14, 15, 16]], result
    print(f"  Result shape: {result.shape}")
    print(f"  Result:\n{result}")
    
    # Index properties and methods
    print(f"\nIndex properties and methods:")
    
    test_slice = ndindex.Slice(2, 10, 3)
    properties = {
        'start': getattr(test_slice, 'start', 'N/A'),
        'stop': getattr(test_slice, 'stop', 'N/A'),
        'step': getattr(test_slice, 'step', 'N/A'),
        'raw': test_slice.raw,
        'ndim': getattr(test_slice, 'ndim', 'N/A')
    }
    
    for prop, value in properties.items():
        print(f"  {prop}: {value}")
    
    # Index validation
    print(f"\nIndex validation:")
    
    # Valid indices
    valid_indices = [
        ndindex.Integer(3),
        ndindex.Slice(0, 5, 1),
        ndindex.Tuple(ndindex.Integer(1), ndindex.Slice(0, 3))
    ]
    
    for idx in valid_indices:
        try:
            # Validate against array shape
            print(f"  {idx}: Valid")
        except Exception as e:
            print(f"  {idx}: Invalid - {e}")
            raise
    
    # Index arithmetic and operations
    print(f"\nIndex operations:")
    
    operations = [
        "Index creation from Python objects",
        "Index validation and normalization", 
        "Index composition and decomposition",
        "Index range calculation and bounds checking",
        "Index conversion between representations"
    ]
    
    for operation in operations:
        print(f"  - {operation}")
    
    print("\nBasic index creation and manipulation completed successfully!")
    
except Exception as e:
    print(f"Error: {e}")
    print("Demonstrating ndindex concepts:")
    concepts = [
        "Index types: Integer, Slice, Tuple, Ellipsis, and Newaxis indices",
        "Index creation: Creating ndindex objects from Python indexing expressions",
        "Index application: Using ndindex objects to index numpy arrays"
    ]
    for concept in concepts:
        print(f"  - {concept}")
    raise  # re-raise: a broken package must fail this example, not print past it

print("Example 1 finished")

## Example 2: Advanced slicing and multidimensional indexing

In [ ]:
import ndindex
import numpy as np

print("Example 2: Advanced slicing and multidimensional indexing")

try:
    print("Exploring advanced slicing capabilities:")
    
    # Create complex multidimensional array
    print(f"\nCreating complex 3D array:")
    arr_3d = np.arange(60).reshape(3, 4, 5)
    print(f"Array shape: {arr_3d.shape}")
    print(f"Array content (first slice):")
    print(arr_3d[0])
    
    # Advanced slicing patterns
    print(f"\nAdvanced slicing patterns:")
    
    slicing_examples = [
        {
            'name': 'Step slicing',
            'index': ndindex.Tuple(ndindex.Slice(None, None, 2), ndindex.Slice(1, None, 2), ndindex.Slice(None)),
            'description': 'Every 2nd element in dim 0, every 2nd from index 1 in dim 1'
        },
        {
            'name': 'Reverse slicing',
            'index': ndindex.Tuple(ndindex.Slice(None, None, -1), ndindex.Slice(None), ndindex.Slice(-3, None)),
            'description': 'Reverse dim 0, normal dim 1, last 3 elements in dim 2'
        },
        {
            'name': 'Mixed indexing',
            'index': ndindex.Tuple(ndindex.Integer(1), ndindex.Slice(None, 3), ndindex.Slice(1, 4, 2)),
            'description': 'Fixed index for dim 0, partial slice for dim 1 and 2'
        },
        {
            'name': 'Ellipsis usage',
            'index': ndindex.Tuple(ndindex.Integer(0), ndindex.ellipsis(), ndindex.Slice(-2, None)),
            'description': 'First element, all middle dimensions, last 2 in final dim'
        }
    ]
    
    for example in slicing_examples:
        try:
            result = arr_3d[example['index'].raw]
            assert example['index'].newshape(arr_3d.shape) == result.shape, (example['index'], result.shape)
            print(f"  {example['name']}: {example['index']}")
            print(f"    Description: {example['description']}")
            print(f"    Result shape: {result.shape}")
            print(f"    Sample values: {result.flat[:5]}")
            print()
        except Exception as slice_error:
            print(f"  {example['name']}: Error - {slice_error}")
            raise
    
    # Index normalization and bounds checking
    print(f"Index normalization and bounds checking:")
    
    # Test various slice normalizations
    shape = (10, 8, 6)
    print(f"Testing normalization for shape {shape}:")
    
    test_slices = [
        ndindex.Slice(-5, -1),      # Negative indices
        ndindex.Slice(None, 15),    # Beyond bounds
        ndindex.Slice(-20, 20),     # Both negative and beyond bounds
        ndindex.Slice(3, 1, -1),    # Negative step
        ndindex.Slice(None, None, -2) # Full reverse with step
    ]
    
    # The canonical form reduce() gives for each slice on an axis of length 10.
    expected_reduced = [slice(5, 9, 1), slice(0, 10, 1), slice(0, 10, 1), slice(3, 1, -1), slice(9, 0, -2)]
    for test_slice, expected in zip(test_slices, expected_reduced):
        try:
            # Reduce the slice against the first dimension and check it selects the same elements
            normalized = test_slice.reduce((shape[0],))
            assert normalized.raw == expected, (test_slice, normalized.raw, expected)
            axis = np.arange(shape[0])
            assert (axis[normalized.raw] == axis[test_slice.raw]).all(), test_slice
            print(f"  Original: {test_slice}")
            print(f"  Raw representation: {test_slice.raw}")
            print(f"  Valid for first dimension")
            print()
        except Exception as norm_error:
            print(f"  {test_slice}: Normalization error - {norm_error}")
            raise
    
    # Complex multidimensional scenarios  
    print(f"Complex multidimensional indexing scenarios:")
    
    # Create larger test array
    large_arr = np.random.random((8, 10, 12, 6))
    print(f"Large array shape: {large_arr.shape}")
    
    complex_indices = [
        {
            'name': 'Subarray extraction',
            'index': ndindex.Tuple(
                ndindex.Slice(2, 6),
                ndindex.Slice(None, None, 3),
                ndindex.Integer(5),
                ndindex.Slice(-3, None)
            ),
            'description': 'Extract specific subarray with mixed indexing'
        },
        {
            'name': 'Sparse sampling',
            'index': ndindex.Tuple(
                ndindex.Slice(None, None, 3),
                ndindex.Slice(1, 8, 2),
                ndindex.Slice(None, None, 4),
                ndindex.Slice(None)
            ),
            'description': 'Sample every 3rd, every 2nd from 1, every 4th'
        },
        {
            'name': 'Edge extraction',
            'index': ndindex.Tuple(
                ndindex.Slice(None, 2),  # First 2
                ndindex.ellipsis(),
                ndindex.Slice(-2, None)  # Last 2
            ),
            'description': 'Extract edges: first 2 in dim 0, last 2 in final dim'
        }
    ]
    
    for example in complex_indices:
        try:
            result = large_arr[example['index'].raw]
            assert example['index'].newshape(large_arr.shape) == result.shape, (example['index'], result.shape)
            print(f"  {example['name']}:")
            print(f"    Index: {example['index']}")
            print(f"    Description: {example['description']}")
            print(f"    Result shape: {result.shape}")
            print(f"    Data type: {result.dtype}")
            print()
        except Exception as complex_error:
            print(f"  {example['name']}: Error - {complex_error}")
            raise
    
    # Index composition and decomposition
    print(f"Index composition features:")
    
    composition_features = [
        "Tuple creation: Combine multiple indices into tuple indices",
        "Index validation: Check index validity against array shapes", 
        "Bounds normalization: Handle negative indices and out-of-bounds cases",
        "Step handling: Proper handling of positive and negative steps",
        "Ellipsis expansion: Automatic expansion of ellipsis to fill dimensions"
    ]
    
    for feature in composition_features:
        print(f"  - {feature}")
    
    print("\\nAdvanced slicing and multidimensional indexing completed successfully!")
    
except Exception as e:
    print(f"Error: {e}")
    print("Demonstrating advanced indexing concepts:")
    concepts = [
        "Complex slicing: Multi-dimensional slicing with steps and negative indices",
        "Index normalization: Handling edge cases and out-of-bounds indices",
        "Tuple indexing: Combining multiple index types for complex array access"
    ]
    for concept in concepts:
        print(f"  - {concept}")
    raise  # re-raise: a broken package must fail this example, not print past it

print("Example 2 finished")

## Example 3: Index transformations and operations

In [ ]:
import ndindex
import numpy as np

print("Example 3: Index transformations and operations")

try:
    print("Exploring index transformations and operations:")
    
    # Index conversion and creation
    print(f"\nIndex conversion from Python objects:")
    
    # Convert various Python indexing expressions to ndindex objects
    python_indices = [
        5,                          # Integer
        slice(2, 8, 2),            # Slice
        (1, slice(None, 5)),       # Tuple with mixed types
        ...,                       # Ellipsis
        np.newaxis,                # Newaxis
        (slice(None), ..., -1),    # Complex tuple
    ]
    
    converted_indices = []
    for i, py_idx in enumerate(python_indices):
        try:
            nd_idx = ndindex.ndindex(py_idx)
            converted_indices.append(nd_idx)
            assert nd_idx.raw == py_idx, (py_idx, nd_idx.raw)
            print(f"  Python: {py_idx}")
            print(f"    ndindex: {nd_idx} (type: {type(nd_idx).__name__})")
            print(f"    Raw: {nd_idx.raw}")
            print()
        except Exception as conv_error:
            print(f"  {py_idx}: Conversion error - {conv_error}")
            raise
    
    assert [type(i).__name__ for i in converted_indices] == ['Integer', 'Slice', 'Tuple', 'ellipsis', 'Newaxis', 'Tuple'], converted_indices
    assert ndindex.Slice(2, 8, 2).as_subindex(ndindex.Slice(0, 10)).raw == slice(2, 7, 2)

    # Index arithmetic and transformations
    print(f"Index arithmetic and transformations:")
    
    # Create base array for testing transformations
    test_array = np.arange(120).reshape(6, 4, 5)
    print(f"Test array shape: {test_array.shape}")
    
    # Demonstrate index operations
    base_index = ndindex.Tuple(ndindex.Slice(1, 5, 2), ndindex.Slice(None), ndindex.Slice(-3, None))
    print(f"Base index: {base_index}")
    
    # Apply transformations
    transformations = [
        {
            'name': 'Identity transformation',
            'operation': lambda idx: idx,
            'description': 'No change to the index'
        },
        {
            'name': 'Dimension reduction',
            'operation': lambda idx: ndindex.Tuple(ndindex.Integer(0), *idx.args[1:]) if isinstance(idx, ndindex.Tuple) else idx,
            'description': 'Fix first dimension to reduce dimensionality'
        },
        {
            'name': 'Step modification',
            'operation': lambda idx: ndindex.Tuple(
                ndindex.Slice(idx.args[0].start, idx.args[0].stop, 1) if isinstance(idx.args[0], ndindex.Slice) else idx.args[0],
                *idx.args[1:]
            ) if isinstance(idx, ndindex.Tuple) else idx,
            'description': 'Change step size of first slice to 1'
        }
    ]
    
    for transform in transformations:
        try:
            transformed = transform['operation'](base_index)
            original_result = test_array[base_index.raw]
            transformed_result = test_array[transformed.raw]
            assert transformed.newshape(test_array.shape) == transformed_result.shape, (transformed, transformed_result.shape)
            
            print(f"  {transform['name']}:")
            print(f"    Description: {transform['description']}")
            print(f"    Original: {base_index} -> shape {original_result.shape}")
            print(f"    Transformed: {transformed} -> shape {transformed_result.shape}")
            print()
        except Exception as transform_error:
            print(f"  {transform['name']}: Error - {transform_error}")
            raise
    
    # Index analysis and properties
    print(f"Index analysis and properties:")
    
    # Analyze different index types
    analysis_indices = [
        ndindex.Slice(2, 10, 3),
        ndindex.Integer(5),
        ndindex.Tuple(ndindex.Slice(None), ndindex.Integer(2), ndindex.Slice(-5, None)),
        ndindex.ellipsis()
    ]
    
    for idx in analysis_indices:
        print(f"  Index: {idx}")
        
        # Extract properties
        properties = {}
        if hasattr(idx, 'start'):
            properties['start'] = idx.start
        if hasattr(idx, 'stop'):
            properties['stop'] = idx.stop
        if hasattr(idx, 'step'):
            properties['step'] = idx.step
        if hasattr(idx, 'ndim'):
            properties['ndim'] = idx.ndim
        if hasattr(idx, 'args'):
            properties['args_count'] = len(idx.args)
        
        properties['raw'] = idx.raw
        properties['type'] = type(idx).__name__
        
        for prop, value in properties.items():
            print(f"    {prop}: {value}")
        print()
    
    # Index composition and decomposition
    print(f"Index composition and decomposition:")
    
    # Create composite index
    component_indices = [
        ndindex.Slice(1, None, 2),
        ndindex.Integer(3),
        ndindex.Slice(None, -1),
        ndindex.Newaxis(),
        ndindex.Slice(-3, None)
    ]
    
    composite = ndindex.Tuple(*component_indices)
    print(f"Composite index: {composite}")
    print(f"Number of components: {len(component_indices)}")
    
    # Decompose the index
    print(f"Components:")
    for i, component in enumerate(component_indices):
        print(f"  Component {i}: {component} ({type(component).__name__})")
        if hasattr(component, 'raw'):
            print(f"    Raw: {component.raw}")
    
    # Index validation scenarios
    print(f"\nIndex validation scenarios:")
    
    validation_cases = [
        {
            'index': ndindex.Slice(0, 10, 1),
            'shape': (15,),
            'expected': 'Valid'
        },
        {
            'index': ndindex.Slice(0, 20, 1),
            'shape': (15,),
            'expected': 'Valid (clipped)'
        },
        {
            'index': ndindex.Integer(5),
            'shape': (10, 8),
            'expected': 'Valid'
        },
        {
            'index': ndindex.Tuple(ndindex.Slice(None), ndindex.Slice(None), ndindex.Integer(0)),
            'shape': (5, 4, 3),
            'expected': 'Valid'
        }
    ]
    
    for case in validation_cases:
        try:
            # Create test array of specified shape
            test_arr = np.zeros(case['shape'])
            result = test_arr[case['index'].raw]
            print(f"  Index {case['index']} on shape {case['shape']}: Valid")
            print(f"    Result shape: {result.shape}")
        except Exception as val_error:
            print(f"  Index {case['index']} on shape {case['shape']}: Invalid - {val_error}")
            raise
    
    print("\\nIndex transformations and operations completed successfully!")
    
except Exception as e:
    print(f"Error: {e}")
    print("Demonstrating index transformation concepts:")
    concepts = [
        "Index conversion: Converting Python indexing expressions to ndindex objects",
        "Index transformation: Modifying and manipulating index objects programmatically",
        "Index analysis: Extracting properties and analyzing index structure"
    ]
    for concept in concepts:
        print(f"  - {concept}")
    raise  # re-raise: a broken package must fail this example, not print past it

print("Example 3 finished")

## Example 4: Array broadcasting and shape manipulation

In [ ]:
import ndindex
import numpy as np

print("Example 4: Array broadcasting and shape manipulation")

try:
    print("Exploring array broadcasting and shape manipulation:")
    
    # Shape manipulation with newaxis
    print(f"\nShape manipulation with newaxis:")
    
    # Create base arrays with different shapes
    arr_1d = np.arange(5)
    arr_2d = np.arange(12).reshape(3, 4)
    arr_3d = np.arange(24).reshape(2, 3, 4)
    
    print(f"Original shapes:")
    print(f"  1D array: {arr_1d.shape} -> {arr_1d}")
    print(f"  2D array: {arr_2d.shape}")
    print(f"  3D array: {arr_3d.shape}")
    
    # Demonstrate newaxis usage
    newaxis_examples = [
        {
            'name': 'Add leading axis',
            'index': ndindex.Tuple(ndindex.Newaxis(), ndindex.Slice(None)),
            'array': arr_1d,
            'description': 'Convert (5,) to (1, 5)'
        },
        {
            'name': 'Add trailing axis',
            'index': ndindex.Tuple(ndindex.Slice(None), ndindex.Newaxis()),
            'array': arr_1d,
            'description': 'Convert (5,) to (5, 1)'
        },
        {
            'name': 'Add middle axis',
            'index': ndindex.Tuple(ndindex.Slice(None), ndindex.Newaxis(), ndindex.Slice(None)),
            'array': arr_2d,
            'description': 'Convert (3, 4) to (3, 1, 4)'
        },
        {
            'name': 'Multiple new axes',
            'index': ndindex.Tuple(ndindex.Newaxis(), ndindex.Slice(None), ndindex.Newaxis(), ndindex.Newaxis()),
            'array': arr_1d,
            'description': 'Convert (5,) to (1, 5, 1, 1)'
        }
    ]
    
    expected_shapes = [(1, 5), (5, 1), (3, 1, 4), (1, 5, 1, 1)]
    for example, expected_shape in zip(newaxis_examples, expected_shapes):
        try:
            result = example['array'][example['index'].raw]
            assert result.shape == expected_shape, (example['name'], result.shape)
            assert example['index'].newshape(example['array'].shape) == expected_shape, example['index']
            print(f"  {example['name']}:")
            print(f"    Index: {example['index']}")
            print(f"    Description: {example['description']}")
            print(f"    Original shape: {example['array'].shape}")
            print(f"    Result shape: {result.shape}")
            print()
        except Exception as newaxis_error:
            print(f"  {example['name']}: Error - {newaxis_error}")
            raise
    
    # Broadcasting scenarios
    print(f"Broadcasting scenarios with ndindex:")
    
    # Create arrays for broadcasting tests
    base_array = np.arange(60).reshape(3, 4, 5)
    broadcast_arrays = {
        'scalar': np.array(10),
        '1d_compatible': np.arange(5),
        '2d_compatible': np.arange(20).reshape(4, 5),
        '3d_subset': np.arange(12).reshape(3, 4, 1)
    }
    
    print(f"Base array shape: {base_array.shape}")
    
    for name, arr in broadcast_arrays.items():
        print(f"  {name} shape: {arr.shape}")
        
        # Test broadcasting compatibility
        try:
            # Use ellipsis to demonstrate broadcasting-friendly indexing
            ellipsis_idx = ndindex.Tuple(ndindex.ellipsis())
            base_subset = base_array[ellipsis_idx.raw]
            
            # Test if shapes are broadcastable
            result_shape = np.broadcast_shapes(base_subset.shape, arr.shape)
            print(f"    Broadcastable to: {result_shape}")
            
        except ValueError as broadcast_error:
            print(f"    Not broadcastable: {broadcast_error}")
    
    # Advanced indexing with broadcasting
    print(f"\nAdvanced indexing with broadcasting implications:")
    
    # Create index patterns that affect broadcasting
    broadcasting_indices = [
        {
            'name': 'Dimension preservation',
            'index': ndindex.Tuple(ndindex.Slice(None), ndindex.Slice(None), ndindex.Slice(None)),
            'description': 'Preserve all dimensions for broadcasting'
        },
        {
            'name': 'Dimension reduction',
            'index': ndindex.Tuple(ndindex.Integer(1), ndindex.Slice(None), ndindex.Slice(None)),
            'description': 'Reduce first dimension, preserve others'
        },
        {
            'name': 'Selective slicing',
            'index': ndindex.Tuple(ndindex.Slice(None, None, 2), ndindex.Slice(1, 3), ndindex.ellipsis()),
            'description': 'Subsample first dim, slice second, preserve rest'
        },
        {
            'name': 'Axis insertion',
            'index': ndindex.Tuple(ndindex.Slice(None), ndindex.Newaxis(), ndindex.Slice(None), ndindex.Slice(None)),
            'description': 'Insert new axis for broadcasting compatibility'
        }
    ]
    
    for example in broadcasting_indices:
        try:
            result = base_array[example['index'].raw]
            print(f"  {example['name']}:")
            print(f"    Index: {example['index']}")
            print(f"    Description: {example['description']}")
            print(f"    Result shape: {result.shape}")
            
            # Test broadcasting with a simple array
            test_broadcast = np.ones((5,))
            try:
                broadcast_result = np.broadcast_shapes(result.shape, test_broadcast.shape)
                print(f"    Broadcasts with (5,): {broadcast_result}")
            except ValueError:
                print(f"    Cannot broadcast with (5,)")
            print()
        except Exception as idx_error:
            print(f"  {example['name']}: Error - {idx_error}")
            raise
    
    # Shape manipulation patterns
    print(f"Shape manipulation patterns:")
    
    manipulation_patterns = [
        "Dimension expansion: Add new axes for broadcasting compatibility",
        "Dimension reduction: Remove specific dimensions through integer indexing",
        "Axis reordering: Rearrange dimensions through advanced tuple indexing",
        "Subarray extraction: Extract compatible subarrays for broadcasting",
        "Broadcasting preparation: Reshape arrays for element-wise operations"
    ]
    
    for pattern in manipulation_patterns:
        print(f"  - {pattern}")
    
    # Practical broadcasting example
    print(f"\nPractical broadcasting example:")
    
    # Create example arrays for a typical broadcasting scenario
    data_3d = np.random.random((4, 6, 8))  # Main data
    weights_2d = np.random.random((6, 8))   # 2D weights
    bias_1d = np.random.random((8,))        # 1D bias
    
    print(f"Data shape: {data_3d.shape}")
    print(f"Weights shape: {weights_2d.shape}")
    print(f"Bias shape: {bias_1d.shape}")
    
    # Use ndindex to prepare arrays for broadcasting
    # Extract subset that's compatible with weights
    data_idx = ndindex.Tuple(ndindex.Slice(None), ndindex.Slice(None), ndindex.Slice(None))
    data_subset = data_3d[data_idx.raw]
    
    # Prepare weights with newaxis for broadcasting
    weights_idx = ndindex.Tuple(ndindex.Newaxis(), ndindex.Slice(None), ndindex.Slice(None))
    weights_broadcast = weights_2d[weights_idx.raw]
    
    # Prepare bias with newaxis for broadcasting
    bias_idx = ndindex.Tuple(ndindex.Newaxis(), ndindex.Newaxis(), ndindex.Slice(None))
    bias_broadcast = bias_1d[bias_idx.raw]
    
    print(f"Broadcasting shapes:")
    print(f"  Data: {data_subset.shape}")
    print(f"  Weights: {weights_broadcast.shape}")
    print(f"  Bias: {bias_broadcast.shape}")
    
    # Verify broadcasting compatibility
    try:
        result_shape = np.broadcast_shapes(data_subset.shape, weights_broadcast.shape, bias_broadcast.shape)
        print(f"  Final broadcasted shape: {result_shape}")
    except ValueError as final_error:
        print(f"  Broadcasting error: {final_error}")
    
    print("\\nArray broadcasting and shape manipulation completed successfully!")
    
except Exception as e:
    print(f"Error: {e}")
    print("Demonstrating broadcasting concepts:")
    concepts = [
        "Newaxis usage: Adding dimensions for broadcasting compatibility",
        "Shape manipulation: Modifying array shapes through indexing",
        "Broadcasting preparation: Setting up arrays for element-wise operations"
    ]
    for concept in concepts:
        print(f"  - {concept}")
    raise  # re-raise: a broken package must fail this example, not print past it

print("Example 4 finished")

## Example 5: Performance optimization and advanced use cases

In [ ]:
import ndindex
import numpy as np
import time

print("Example 5: Performance optimization and advanced use cases")

try:
    print("Exploring performance optimization and advanced use cases:")
    
    # Performance comparison: ndindex vs native Python indexing
    print(f"\nPerformance comparison:")
    
    # Create large test array
    large_array = np.random.random((1000, 500, 100))
    print(f"Test array shape: {large_array.shape}")
    
    # Define indexing operations
    test_cases = [
        {
            'name': 'Simple slice',
            'python_index': (slice(100, 800, 5), slice(None), slice(10, 90, 2)),
            'description': 'Basic slicing with steps'
        },
        {
            'name': 'Complex tuple',
            'python_index': (slice(None, None, 10), slice(50, 450), slice(-50, None)),
            'description': 'Mixed slicing with negative indices'
        },
        {
            'name': 'Dimension reduction',
            'python_index': (500, slice(None), slice(None, None, 3)),
            'description': 'Integer index with slicing'
        }
    ]
    
    for case in test_cases:
        print(f"  {case['name']} - {case['description']}:")
        
        # Convert to ndindex
        nd_idx = ndindex.ndindex(case['python_index'])
        
        # Time native Python indexing
        start_time = time.time()
        for _ in range(100):
            result_python = large_array[case['python_index']]
        python_time = time.time() - start_time
        
        # Time ndindex indexing
        start_time = time.time()
        for _ in range(100):
            result_ndindex = large_array[nd_idx.raw]
        ndindex_time = time.time() - start_time
        
        print(f"    Python indexing: {python_time:.4f} seconds (100 iterations)")
        print(f"    ndindex indexing: {ndindex_time:.4f} seconds (100 iterations)")
        assert result_python.shape == result_ndindex.shape, (result_python.shape, result_ndindex.shape)
        assert np.array_equal(result_python, result_ndindex)
        assert nd_idx.newshape(large_array.shape) == result_python.shape, nd_idx
        print(f"    Result shapes match: {result_python.shape == result_ndindex.shape}")
        print()
    
    # Advanced use cases
    print(f"Advanced use cases:")
    
    # Dynamic index generation
    print(f"  Dynamic index generation:")
    
    def generate_sampling_index(shape, sample_rate=0.3, dimensions_to_sample=None):
        """Generate ndindex for random sampling of array"""
        if dimensions_to_sample is None:
            dimensions_to_sample = list(range(len(shape)))
        
        indices = []
        for i, dim_size in enumerate(shape):
            if i in dimensions_to_sample:
                # Sample this dimension
                step = max(1, int(1 / sample_rate))
                indices.append(ndindex.Slice(None, None, step))
            else:
                # Keep full dimension
                indices.append(ndindex.Slice(None))
        
        return ndindex.Tuple(*indices)
    
    # Generate sampling indices for different scenarios
    sampling_scenarios = [
        {'sample_rate': 0.5, 'dimensions': [0, 2], 'description': 'Sample 50% of dims 0 and 2'},
        {'sample_rate': 0.2, 'dimensions': [1], 'description': 'Sample 20% of dim 1 only'},
        {'sample_rate': 0.3, 'dimensions': None, 'description': 'Sample 30% of all dimensions'}
    ]
    
    test_shape = (200, 150, 80)
    test_array = np.random.random(test_shape)
    
    for scenario in sampling_scenarios:
        sample_idx = generate_sampling_index(
            test_shape, 
            scenario['sample_rate'], 
            scenario['dimensions']
        )
        
        result = test_array[sample_idx.raw]
        reduction_factor = np.prod(test_shape) / np.prod(result.shape)
        
        print(f"    {scenario['description']}:")
        print(f"      Generated index: {sample_idx}")
        print(f"      Original shape: {test_shape}")
        print(f"      Sampled shape: {result.shape}")
        print(f"      Data reduction: {reduction_factor:.1f}x")
        print()
    
    # Index validation and error handling
    print(f"Index validation and error handling:")
    
    validation_examples = [
        {
            'name': 'Out of bounds handling',
            'index_func': lambda: ndindex.Slice(0, 1000),  # Beyond array bounds
            'array_shape': (100,),
            'expected': 'Should handle gracefully'
        },
        {
            'name': 'Negative step validation',
            'index_func': lambda: ndindex.Slice(50, 10, -2),
            'array_shape': (100,),
            'expected': 'Valid reverse slice'
        },
        {
            'name': 'Complex tuple validation',
            'index_func': lambda: ndindex.Tuple(
                ndindex.Slice(None), 
                ndindex.Integer(5), 
                ndindex.ellipsis(),
                ndindex.Slice(-10, None)
            ),
            'array_shape': (20, 10, 5, 8),
            'expected': 'Valid complex indexing'
        }
    ]
    
    for example in validation_examples:
        try:
            test_array = np.zeros(example['array_shape'])
            test_index = example['index_func']()
            result = test_array[test_index.raw]
            
            print(f"  {example['name']}: Valid")
            print(f"    Index: {test_index}")
            print(f"    Array shape: {example['array_shape']}")
            print(f"    Result shape: {result.shape}")
            print(f"    Expected: {example['expected']}")
            
        except Exception as validation_error:
            print(f"  {example['name']}: Error - {validation_error}")
            raise
        print()
    
    # Real-world application scenarios
    print(f"Real-world application scenarios:")
    
    applications = [
        {
            'domain': 'Scientific computing',
            'use_cases': [
                'Multi-dimensional data slicing for analysis',
                'Efficient subarray extraction for processing',
                'Dynamic indexing based on experimental parameters'
            ]
        },
        {
            'domain': 'Machine learning',
            'use_cases': [
                'Batch processing with dynamic batch sizes',
                'Feature selection through advanced indexing',
                'Data augmentation through systematic sampling'
            ]
        },
        {
            'domain': 'Image/signal processing',
            'use_cases': [
                'Region of interest extraction',
                'Multi-channel data manipulation',
                'Temporal data windowing and analysis'
            ]
        },
        {
            'domain': 'Data analysis',
            'use_cases': [
                'Time series subsampling and analysis',
                'Multi-dimensional data exploration',
                'Efficient data filtering and selection'
            ]
        }
    ]
    
    for app in applications:
        print(f"  {app['domain']}:")
        for use_case in app['use_cases']:
            print(f"    - {use_case}")
        print()
    
    # Performance optimization tips
    print(f"Performance optimization tips:")
    
    optimization_tips = [
        "Pre-create indices: Reuse ndindex objects for repeated operations",
        "Validate once: Check index validity before loops, not during",
        "Use views: Leverage NumPy's view mechanism through proper slicing",
        "Minimize copies: Design indices to work with array views when possible",
        "Batch operations: Process multiple indices together when feasible"
    ]
    
    for tip in optimization_tips:
        print(f"  - {tip}")
    
    # Library integration benefits
    print(f"\nLibrary integration benefits:")
    
    integration_benefits = [
        "Type safety: Static analysis of indexing operations",
        "Index algebra: Mathematical operations on index objects",
        "Serialization: Save and load complex indexing patterns",
        "Debugging: Clear representation of complex indexing operations",
        "Documentation: Self-documenting indexing code"
    ]
    
    for benefit in integration_benefits:
        print(f"  - {benefit}")
    
    print("\\nPerformance optimization and advanced use cases completed successfully!")
    
except Exception as e:
    print(f"Error: {e}")
    print("Demonstrating advanced concepts:")
    concepts = [
        "Performance optimization: Efficient indexing for large arrays",
        "Dynamic indexing: Programmatic generation of index patterns",
        "Real-world applications: Practical use cases across different domains"
    ]
    for concept in concepts:
        print(f"  - {concept}")
    raise  # re-raise: a broken package must fail this example, not print past it

print("Example 5 finished")